# Project Echo :: Error Analysis

The following notebook contains a fairly basic implementation for error analysis with regards to the DataBytes Project Echo Engine. We're taking a minimal pass starting from a stratified manifest, to batch inference, to summary analysis.

Edit the CONFIG cell, then run top-to-bottom.

In [1]:
from pathlib import Path

# EDIT THESE TWO 
INTEGRATION_DIR = Path("PATH/TO/MODEL")  # Contains efficientnetv2_predictor.py and _trained_models/
DATA_ROOT = Path("PATH/TO/DATA")  # Folder containing one subfolder per species

MODEL_DIR = INTEGRATION_DIR / "_trained_models"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLES_PER_CLASS = 20  # cap; classes with fewer files use what they have
SEED = 42
AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg")


## Build a stratified manifest

Sample up to N files per class so every class is represented, not just the alphabetically-early ones. An early problem with testing was verification of the pipeline on only a small subset of samples. This attempts to rectify that.

In [2]:
import json, random
import pandas as pd

random.seed(SEED)

classes = json.loads((MODEL_DIR / "class_mapping.json").read_text())["classes"]
on_disk = {d.name for d in DATA_ROOT.iterdir() if d.is_dir()}
usable = sorted(set(classes) & on_disk)

rows = []
for c in usable:
    files = [p for p in (DATA_ROOT / c).rglob("*") if p.suffix.lower() in AUDIO_EXTS]
    for f in random.sample(files, min(SAMPLES_PER_CLASS, len(files))):
        rows.append({"file_path": str(f), "true_label": c, "n_available": len(files)})

manifest = pd.DataFrame(rows)
print(f"{len(manifest)} files across {manifest['true_label'].nunique()} classes")
manifest.head()


2166 files across 123 classes


,file_path,true_label,n_available
0,/Users/jack/Documents/Deakin/SIT374/Tasks/data...,Acanthiza chrysorrhoa,34
1,/Users/jack/Documents/Deakin/SIT374/Tasks/data...,Acanthiza chrysorrhoa,34
2,/Users/jack/Documents/Deakin/SIT374/Tasks/data...,Acanthiza chrysorrhoa,34
3,/Users/jack/Documents/Deakin/SIT374/Tasks/data...,Acanthiza chrysorrhoa,34
4,/Users/jack/Documents/Deakin/SIT374/Tasks/data...,Acanthiza chrysorrhoa,34


## Run inference

Import `predict_audio` from the existing predictor (see the github) so we're testing exactly what production runs. Takes a few minutes as I'm personally bound be CPU. Adapt this to CUDA if you so desire.

In [4]:
import sys
sys.path.insert(0, str(INTEGRATION_DIR))
from efficientnetv2_predictor import load_predictor_bundle, predict_audio

import warnings
warnings.filterwarnings("ignore")

model, class_mapping, preprocess_config, device = load_predictor_bundle(
    model_path=MODEL_DIR / "efficientnetv2_project_echo.pt",
    class_mapping_path=MODEL_DIR / "class_mapping.json",
    preprocess_config_path=MODEL_DIR / "preprocess_config.json",
)
print(f"Model loaded on {device}")

results = []
for i, row in enumerate(manifest.itertuples(), 1):
    r = predict_audio(row.file_path, model, class_mapping, preprocess_config, device)
    top5 = [(p["label"], p["confidence"]) for p in r["top_predictions"]]
    results.append({
        "file_path": row.file_path,
        "true_label": row.true_label,
        "predicted_label": r["predicted_label"],
        "confidence": r["confidence"],
        "top5": top5,
        "is_correct": r["predicted_label"] == row.true_label,
        "is_top5": row.true_label in [t[0] for t in top5],
    })
    if i % 100 == 0:
        print(f"  {i}/{len(manifest)}")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"Done. {len(df)} predictions saved.")


Model loaded on cpu
  100/2166
  200/2166
  300/2166
  400/2166
  500/2166
  600/2166
  700/2166
  800/2166
  900/2166
  1000/2166
  1100/2166
  1200/2166
  1300/2166
  1400/2166
  1500/2166
  1600/2166
  1700/2166
  1800/2166
  1900/2166
  2000/2166
  2100/2166
Done. 2166 predictions saved.


## Headline numbers

These are the very core essentials of what we're chasing. This section deserves to be built up a little stronger but for now we focus on that absolute minimum requirement.

In [6]:
print(f"Top-1 accuracy: {df['is_correct'].mean():.1%}")
print(f"Top-5 accuracy: {df['is_top5'].mean():.1%}")
print(f"Mean confidence when correct: {df.loc[df['is_correct'], 'confidence'].mean():.3f}")
print(f"Mean confidence when wrong: {df.loc[~df['is_correct'], 'confidence'].mean():.3f}")


Top-1 accuracy: 95.8%
Top-5 accuracy: 98.3%
Mean confidence when correct: 0.911
Mean confidence when wrong: 0.448


## Worst classes

Per-class F1, sorted ascending. The bottom of this table is ideally where Sprint 3 should focus.

In [7]:
from sklearn.metrics import precision_recall_fscore_support

labels = sorted(df["true_label"].unique())
p, r, f1, support = precision_recall_fscore_support(
    df["true_label"], df["predicted_label"], labels=labels, zero_division=0
)
metrics = pd.DataFrame({"class": labels, "support": support,
                       "precision": p, "recall": r, "f1": f1})
metrics = metrics.sort_values("f1").reset_index(drop=True)
metrics.to_csv(OUTPUT_DIR / "class_metrics.csv", index=False)
print(metrics.head(15).to_string(index=False))


                       class  support  precision   recall       f1
        Rhipidura leucophrys       20   0.593750 0.950000 0.730769
         Rhipidura albiscapa       20   0.782609 0.900000 0.837209
Acanthorhynchus tenuirostris       20   0.782609 0.900000 0.837209
     Anhinga novaehollandiae        9   1.000000 0.777778 0.875000
                 Fulica atra       14   1.000000 0.785714 0.880000
     Colluricincla harmonica       20   0.800000 1.000000 0.888889
              Vanellus miles       20   1.000000 0.800000 0.888889
      Philemon citreogularis       20   0.800000 1.000000 0.888889
            Lalage leucomela       20   0.944444 0.850000 0.894737
                      sheowl       20   0.900000 0.900000 0.900000
         Haliastur sphenurus       20   0.900000 0.900000 0.900000
           Chenonetta jubata       15   0.875000 0.933333 0.903226
        Spilopelia chinensis       10   0.833333 1.000000 0.909091
       Falcunculus frontatus       13   1.000000 0.846154 0.91

## Problem classes

Which classes does the model over-predict? These are the ones inflating false positives. We need a good way to sort these if we're going to have a model that runs effectively.

In [8]:
errors = df[~df["is_correct"]]
print("Top over-predicted classes (FP source):")
print(errors["predicted_label"].value_counts().head(10).to_string())
print("\nTop misclassified classes (FN source):")
print(errors["true_label"].value_counts().head(10).to_string())


Top over-predicted classes (FP source):
predicted_label
Rhipidura leucophrys            13
Rhipidura albiscapa              5
Colluricincla harmonica          5
Philemon citreogularis           5
Acanthorhynchus tenuirostris     5
Conopophila albogularis          3
Manorina melanophrys             3
Acanthiza reguloides             3
sheowl                           2
Melithreptus gularis             2

Top misclassified classes (FN source):
true_label
Vanellus miles               4
Eurystomus orientalis        3
Melithreptus brevirostris    3
Fulica atra                  3
Phaps elegans                3
Lalage leucomela             3
Falco berigora               2
Falcunculus frontatus        2
Haliastur sphenurus          2
Megapodius reinwardt         2


## Confidence threshold

How does precision change with the display threshold?

In [9]:
import numpy as np
rows = []
for t in np.arange(0.5, 1.0, 0.05):
    kept = df[df["confidence"] >= t]
    rows.append({"threshold": round(t, 2),
                "share_kept": len(kept) / len(df),
                "precision": kept["is_correct"].mean() if len(kept) else None})
print(pd.DataFrame(rows).to_string(index=False))


 threshold  share_kept  precision
      0.50    0.938135   0.984744
      0.55    0.927054   0.988546
      0.60    0.913666   0.990904
      0.65    0.900739   0.991799
      0.70    0.889658   0.992735
      0.75    0.867959   0.993617
      0.80    0.838873   0.995047
      0.85    0.800092   0.995384
      0.90    0.733149   0.996222
      0.95    0.596491   0.996904
